
# M2M-100 (zh→en) — Baseline TF Training (Wuxia Domain)

**TFG Anonymous – Baseline NMT (M2M-100)**  
This notebook trains and evaluates to model **M2M-100** ("facebook/m2m100_418M") using to dataset of the dominio **wuxia** (Chinese->English) already preparado in format `datasets` (HF).






## 1) Preparation of the environment

In [ ]:

import os, random, math
import numpy as np

import torch
print("CUDA disponible:", torch.cuda.is_available())
print("Number of GPUs:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Name of the GPU:", torch.cuda.get_device_name(0))



> **Requirements of the dataset**: directory HF Datasets with *splits* `train`, `validation`, `test` and columns `zh` (Chinese) and `en` (English):  
> `processed_data/wuxia_zh_en_clean/`

In [ ]:
# Configuration of carpetas for repository LOCAL
from pathlib import Path
BASE_DIR = Path.cwd().parent
BASE_DIR.mkdir(exist_ok=True)

# Structure of the repository
for sub in ["training", "models", "processed_data"]:
    (BASE_DIR / sub).mkdir(parents=True, exist_ok=True)

print("Base:", BASE_DIR.resolve())
print("Structure created (if not existed):")
for p in ["trainign", "models", "proccesed data"]:
    print(" -", (BASE_DIR / p).resolve())

# Score: the dataset must existir in: CORPUS/proccesed data/wuxia_zh_en_clean


## 2) Configuration

In [ ]:
from dataclasses import dataclass

@dataclass
class Config:
    # Paths in local
    dataset_dir: Path  = BASE_DIR / "processed_data" / "wuxia_zh_en_clean"   # <- directory with dataset HuggingFace 
    output_dir: Path   = BASE_DIR / "models" / "m2m100_wuxia"              # <- here is will be saved the runs/models
    ckpt_dir: Path     = BASE_DIR / "checkpoints"                          # <- checkpoints during training
    training_dir: Path = BASE_DIR / "training"         

    # Columns of the dataset
    src_col: str = "zh"
    tgt_col: str = "en"

    # Idiomas M2M-100
    src_lang: str = "zh"
    tgt_lang: str = "en"

    # Model
    model_ckpt: str = "facebook/m2m100_418M"

    # Training
    seed: int = 42
    max_source_length: int = 128
    max_target_length: int = 128
    batch_size: int = 16
    epochs: int = 10
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    early_stopping_patience: int = 3

    fraction: float = 1

cfg = Config()
print(cfg)


In [ ]:
import random, numpy as np, os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo:", device)

# Semillas 
torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)
random.seed(cfg.seed)
os.environ["PYTHONHASHSEED"] = str(cfg.seed)


if device.type == "cuda":
    torch.cuda.manual_seed_all(cfg.seed)
    # For reproducibilidad
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("Semillas fijadas and backend configured.")


## 3) Load dataset (Hugging Face Datasets)

In [ ]:

from datasets import load_from_disk, DatasetDict

assert os.path.isdir(cfg.dataset_dir), f"The dataset was not found at: {cfg.dataset_dir}"
raw_ds: DatasetDict = load_from_disk(cfg.dataset_dir)
print(raw_ds)

# Validar columns
def _check_cols(ds, src_col, tgt_col, split):
    cols = ds.column_names
    assert src_col in cols and tgt_col in cols, f"El split '{split}' must contener columns '{src_col}' y '{tgt_col}'. Columns: {cols}"

for split in ["train", "validation", "test"]:
    assert split in raw_ds, f"Falta el split '{split}' en el dataset."
    _check_cols(raw_ds[split], cfg.src_col, cfg.tgt_col, split)

# testeos
def take_fraction(ds, frac, seed=42):
    if frac >= 1.0:
        return ds
    n = max(1, int(len(ds) * frac))
    return ds.shuffle(seed=seed).select(range(n))

train_ds = take_fraction(raw_ds["train"], cfg.fraction, seed=cfg.seed)
val_ds   = take_fraction(raw_ds["validation"], cfg.fraction, seed=cfg.seed)
test_ds  = take_fraction(raw_ds["test"], cfg.fraction, seed=cfg.seed)

print(train_ds[:2])
print(f"Tam. train/val/test (fraction={cfg.fraction}):", len(train_ds), len(val_ds), len(test_ds))


## 4) Load tokenizador and model M2M-100 (zh→en)


In [ ]:
from transformers import M2M100Tokenizer, M2M100ForConditionalGeneration

tokenizer = M2M100Tokenizer.from_pretrained(cfg.model_ckpt)
model = M2M100ForConditionalGeneration.from_pretrained(cfg.model_ckpt)
tokenizer.src_lang = cfg.src_lang
model.config.forced_bos_token_id = tokenizer.get_lang_id(cfg.tgt_lang)
model.to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {cfg.model_ckpt}")
print(f"Parameters total: {n_params:,}")

# testeo
sample = {
    'zh': '江湖夜雨十年灯。',
    'en': 'Ten years of lamps in the night rain of the jianghu.'
}
inputs = tokenizer(sample['zh'], return_tensors='pt').to(device)
with torch.no_grad():
    out = model.generate(
        **inputs,
        max_length=cfg.max_target_length,
        num_beams=4, early_stopping=True,
        forced_bos_token_id=tokenizer.get_lang_id(cfg.tgt_lang)
    )
pred = tokenizer.decode(out[0], skip_special_tokens=True)
print('='*80)
print('ZH:', sample['zh'])
print('IN (ref):', sample['en'])
print('IN (pred):', pred)


## 5) Preprocesamiento and tokenization

In [ ]:
# Function for tokenize M2M-100
def preprocess_function(examples):
    # important fijar idiomas in each llamada
    tokenizer.src_lang = cfg.src_lang         # source 
    # For tokenize labels with text_target is uses tgt_lang
    try:
        tokenizer.tgt_lang = cfg.tgt_lang     # target
    except Exception:
        pass

    
    model_inputs = tokenizer(
        examples[cfg.src_col],
        max_length=cfg.max_source_length,
        padding=False,
        truncation=True
    )

    try:
        labels = tokenizer(
            text_target=examples[cfg.tgt_col],
            max_length=cfg.max_target_length,
            padding=False,
            truncation=True
        )
    except TypeError:
        # Compatibility with transformers older
        with tokenizer.as_target_tokenizer():
            labels = tokenizer(
                examples[cfg.tgt_col],
                max_length=cfg.max_target_length,
                padding=False,
                truncation=True
            )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Apply tokenization to all the dataset
tokenized_datasets = raw_ds.map(
    preprocess_function,
    batched=True,
    remove_columns=raw_ds["train"].column_names
)

train_ds = take_fraction(tokenized_datasets["train"], cfg.fraction, seed=cfg.seed)
val_ds   = take_fraction(tokenized_datasets["validation"], cfg.fraction, seed=cfg.seed)
test_ds  = take_fraction(tokenized_datasets["test"], cfg.fraction, seed=cfg.seed)

print(train_ds[0])


## 6) Data collator

In [ ]:
from transformers import DataCollatorForSeq2Seq

# Data collator for "facebook/m2m100_418M"
# Is encarga of align dynamically the sequences and create batches listos for the model
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding="longest",   
    return_tensors="pt"
)

# Example of batch
batch = data_collator([train_ds[i] for i in range(2)])
for k, v in batch.items():
    print(f"{k}: shape={v.shape}, dtype={v.dtype}")

## 7) Configuration of training PyTorch + Seq2SeqTrainer



> **By default**          
> **Optimizador**: `AdamW` (with LR=2e-5, weight decay=0.01) of `Seq2SeqTrainer`   
> **Loss**: `CrossEntropyLoss` (token-level) of AutoModelForSeq2SeqLM (`M2M100`)  





In [ ]:

from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

# Directory for results
run_dir = cfg.output_dir
run_dir.mkdir(parents=True, exist_ok=True)


training_args = Seq2SeqTrainingArguments(
    output_dir=str(run_dir),
    overwrite_output_dir=True,
    eval_strategy="epoch",                  # Evaluar to the final of each epoch
    save_strategy="epoch",                  # Save checkpoint by epoch
    save_total_limit=3,                      # Max number of checkpoints saved
    learning_rate=cfg.learning_rate,
    num_train_epochs=cfg.epochs,
    per_device_train_batch_size=cfg.batch_size,
    per_device_eval_batch_size=cfg.batch_size,
    weight_decay=cfg.weight_decay,
    logging_dir=str(run_dir / "logs"),
    logging_strategy="steps",
    logging_steps=50,
    predict_with_generate=True,              # Generate sequences in validation
    fp16=torch.cuda.is_available(),          # Precision mixta if there are GPU
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False
)

# Trainer for Seq2Seq
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator
)

print(" Seq2SeqTrainer configured (PyTorch).")


## 8) Training

In [ ]:

from transformers import EarlyStoppingCallback
import json

# Add callback of early stopping
trainer.add_callback(EarlyStoppingCallback(
    early_stopping_patience=cfg.early_stopping_patience,  # number of evaluations without improvement
    early_stopping_threshold=0.0
))

# Entrenar
train_result = trainer.train()

# Save model final and tokenizer
trainer.save_model(cfg.output_dir)
tokenizer.save_pretrained(cfg.output_dir)

# Save metrics of training
metrics = train_result.metrics
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)
trainer.save_state()

# Save configuration and results in JSON 
run_info = {
    "model_name": cfg.model_ckpt,
    "epochs": cfg.epochs,
    "batch_size": cfg.batch_size,
    "learning_rate": cfg.learning_rate,
    "weight_decay": cfg.weight_decay,
    "train_size": len(train_ds),
    "val_size": len(val_ds),
    "metrics": metrics
}

with open(cfg.output_dir / "run_info.json", "w", encoding="utf-8") as f:
    json.dump(run_info, f, indent=4, ensure_ascii=False)

print(" Training finished, model and artefactos saved in:", cfg.output_dir)
